# Licenciatura en Ciencia de Datos
## Calidad y Preprocesamiento de Datos

---

### **Proyecto Final — Limpieza y Golden Record**

**Equipo**

**Castrillo Cruz Karen Arlet**

**Pérez Aguiar Oropeza Gabriel Emiliano**

**Ramos González Nadia**

**Rueda Reyes Fabián**

**Torres Pasión Angel Isaac**


### **Objetivo**

Limpiar y normalizar las bases INE y BANAVIM, reparar problemas de codificación (mojibake), y construir el Golden Record de la base INE por persona. Este notebook exporta tres archivos que usa `fusion.ipynb`:

- `ine_incidencias_2020_2022.csv` — INE limpio y filtrado
- `banavim_fix_2020_2022.pkl` — BANAVIM con texto reparado
- `df_golden.pkl` — Golden Record consolidado por individuo


## 0. Ingesta de Datos


In [1]:
# Instalación de dependencias
!pip install unidecode openpyxl ftfy --quiet


In [2]:
import pandas as pd
import numpy as np
import re
import pickle
from pathlib import Path
from unicodedata import normalize
from unidecode import unidecode

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from ftfy import fix_text

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style='whitegrid')

print('Entorno listo')


Entorno listo


In [3]:
# Rutas base — funciona en Windows y Linux/macOS
DATA_DIR = Path('..') / 'Data'
PKL_DIR  = Path('.')   # los pkl viven en el mismo directorio que el notebook

DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'fusion_outputs').mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR.resolve())
print('PKL_DIR :', PKL_DIR.resolve())


DATA_DIR: /home/emi/Desktop/Calidad-y-Preprocesamiento-de-Datos---Registro-Publico-de-Agresores/Data
PKL_DIR : /home/emi/Desktop/Calidad-y-Preprocesamiento-de-Datos---Registro-Publico-de-Agresores/Código


In [4]:
# 0.1 INE — Registro Nacional de Personas Sancionadas
df_ine_raw = pd.read_excel(
    DATA_DIR / 'Registro-nacional-de-personas-sancionadas (INE).xlsx'
)
df_ine_raw.columns = df_ine_raw.columns.str.strip()
print(f'INE crudo: {df_ine_raw.shape}')

# 0.2 BANAVIM — desde pkl pregenerados para evitar parsear los Excel originales
def leer_banavim_agresores(path: Path, año: int) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=f'AGRESORES {año}', header=0)
    df.columns = df.iloc[0]
    df = df.drop(index=0).reset_index(drop=True)
    df.columns = df.columns.str.strip()
    df['año_base'] = año
    return df

# bv_2020 = leer_banavim_agresores(DATA_DIR / '2020.xlsx', 2020)
# bv_2021 = leer_banavim_agresores(DATA_DIR / '2021.xlsx', 2021)
# bv_2022 = leer_banavim_agresores(DATA_DIR / '2022.xlsx', 2022)

for año in [2020, 2021, 2022]:
    with open(PKL_DIR / f'bv_{año}.pkl', 'rb') as f:
        globals()[f'bv_{año}'] = pickle.load(f)
    print(f'BANAVIM {año}: {globals()[f"bv_{año}"].shape}')


INE crudo: (530, 25)
BANAVIM 2020: (265760, 28)
BANAVIM 2021: (285984, 28)
BANAVIM 2022: (254348, 28)


In [5]:
# Vista previa
print('INE (últimas 5 filas)')
display(df_ine_raw.tail(5))
print('BANAVIM 2020 (primeras 3 filas)')
display(bv_2020.head(3))

INE (últimas 5 filas)


,Nombre,"Calidad, cargo o profesión del sujeto infractor",Sexo,Ámbito Territorial,Entidad Federativa,Municipio,Número De Expediente,Relación Con La Víctima,Incidencia,Órgano Resolutor,Fecha De La Resolución,Conducta,Sanción,Permanencia,Reincidencia De La Conducta,Resolución Penal,Analizó Modo Honesto De Vivir,Cumple Modo Honesto De Vivir,Perteneciente a,Documento Enlace,Enlace Utce Temporalidad,Interseccion de la víctima,Tipo de violencia,Modalidad de violencia,Medidas de reparacion
525,YOLANDA ADELAIDA SANTOS MONTAÑO,NaN,Mujer,Municipal,Oaxaca,San Jacinto Amilpas,JDC/133/2020,Pares,No aplica,TE,11/06/2021,"Negativa a restituir a la víctima en el pleno goce de sus derechos políticos electorales, al omi...",Ninguna,29/09/2027,Sí,No,Sí,No,NaN,https://repositoriodocumental.ine.mx/xmlui/bitstream/handle/123456789/131519/JDC-133-2020-TEEO.pdf,NaN,NaN,"tipo_conducta type=""Collection(Edm.String)"">_x000d_\n <element>No lo precisa</element>_x000d_\n...",NaN,"medidas_reparacion type=""Collection(Edm.String)"">_x000d_\n <element>Medidas de no repetición</e..."
526,YOLANDA ADELAIDA SANTOS MONTAÑO,NaN,Mujer,Municipal,Oaxaca,San Jacinto Amilpas,JDC/143/2020,Pares,No aplica,TE,11/06/2021,Omisión de pago de dietas a la víctima y de convocarla a sesiones de cabildo,Ninguna,29/09/2027,Sí,No,Sí,No,NaN,https://repositoriodocumental.ine.mx/xmlui/bitstream/handle/123456789/131520/JDC-143-2020-TEEO.pdf,NaN,NaN,"tipo_conducta type=""Collection(Edm.String)"">_x000d_\n <element>No lo precisa</element>_x000d_\n...",NaN,"medidas_reparacion type=""Collection(Edm.String)"">_x000d_\n <element>Medidas de no repetición</e..."
527,YOLANDA ADELAIDA SANTOS MONTAÑO,NaN,Mujer,Municipal,Oaxaca,San Jacinto Amilpas,SUP-REC-117/2022,Pares,"Pérdida del modo honesto de vivir para los próximos procesos electorales federales, locales y mu...",TEPJF SS,04/05/2022,Revictimización derivada del incumplimiento de las medidas ordenadas ante la existencia de la VP...,Perdida del modo honesto de vivir,05/05/2028,Sí,No,Sí,No,NaN,https://repositoriodocumental.ine.mx/xmlui/bitstream/handle/123456789/134552/SUP-REC-117-2022.pdf,NaN,NaN,"tipo_conducta type=""Collection(Edm.String)"">_x000d_\n <element>No lo precisa</element>_x000d_\n...",NaN,"medidas_reparacion type=""Collection(Edm.String)"">_x000d_\n <element>No impone</element>_x000d_\..."
528,YOSHIO CÉSAR RAMÍREZ CASTILLO,NaN,Hombre,Municipal,Oaxaca,Ocotlán de Morelos,JDC/05/2024 Y JDC/96/2024 ACUMULADOS,Pares,No aplica,TE,20/09/2024,"Conducta consistente en limitar el acceso a la información financiera, además de no proporcionar...",Amonestación pública,2029-06-04T00:00:00,No,No,No,NaN,NaN,https://teeo.mx/images/sentencias/JDC-05-2024-1.pdf,NaN,NaN,"tipo_conducta type=""Collection(Edm.String)"">_x000d_\n <element>Verbal</element>_x000d_\n <elem...",NaN,"medidas_reparacion type=""Collection(Edm.String)"">_x000d_\n <element>Disculpa pública</element>_..."
529,YURIDIA PINEDA ORDAZ,NaN,Mujer,Municipal,Oaxaca,El Espinal,JDC/797/2022,Pares,No aplica,TE,08/02/2023,"Invisibilizar a la víctima al limitarla de proporcionarle materiales de oficina, documentación y...",Ninguna,2029-06-01T00:00:00,Sí,No,Sí,Sí,NaN,https://teeo.mx/images/sentencias/JDC-797-2022.pdf,NaN,NaN,"tipo_conducta type=""Collection(Edm.String)"">_x000d_\n <element>Psicológica</element>_x000d_\n ...",NaN,"medidas_reparacion type=""Collection(Edm.String)"">_x000d_\n <element>Disculpa pública</element>_..."


BANAVIM 2020 (primeras 3 filas)


,Identificador Único,Edad del Agresor,Sexo,Escolaridad,Estado Civil,Fecha de registro,Estado donde reside,Municipio donde reside,Relación o vículo con la víctima,Conoce al Agresor,Durante la agresión efectos droga,Cual Droga,Droga_Alcohol,Droga_DrogaPorIndicación Médica,Droga_Drogas Ilegales,La Consume Manera Cotidiana,Posee_algun tipo de arma,Portaba Dicha Arma,Chacos,Macanas,OtraArmaBlanca,ObjetoPunzoCortante,Machete,Proyectil,ArmaFuegoCorta,ArmaFuegoLarga,OtraFuegoLarga,año_base
0,0128900022-2,34,Hombre,No identificado,NaN,2020-09-03 08:59:29,Aguascalientes,Aguascalientes,Ex pareja,SI,SE DESCONOCE,NaN,0,0,0,NaN,SE DESCONOCE,NaN,0,0,0,0,0,0,0,0,0,2020
1,0128900106-2,50,Hombre,No identificado,NaN,2020-10-20 09:25:05,Aguascalientes,Aguascalientes,CÃ³nyuge o pareja,SI,SE DESCONOCE,NaN,0,0,0,NaN,SE DESCONOCE,NaN,0,0,0,0,0,0,0,0,0,2020
2,0128900132-2,58,Hombre,No identificado,UniÃ³n libre,2020-07-02 08:07:24,Aguascalientes,Aguascalientes,CÃ³nyuge o pareja,SI,SE DESCONOCE,NaN,0,0,0,NaN,SE DESCONOCE,NaN,0,0,0,0,0,0,0,0,0,2020


## 2. Limpieza y Normalización

### 2.1 INE

In [6]:
# 2.1 Funciones de normalización

MAPA_ESTADOS = {
    'CIUDAD DE MEXICO': 'CDMX',
    'D.F.': 'CDMX',
    'MEXICO': 'ESTADO DE MEXICO',
    'EDO MEX': 'ESTADO DE MEXICO',
    'VERACRUZ DE IGNACIO DE LA LLAVE': 'VERACRUZ',
    'MICHOACAN DE OCAMPO': 'MICHOACAN',
    'COAHUILA DE ZARAGOZA': 'COAHUILA'
}

def normalizar_nombre(texto):
    if pd.isna(texto) or not str(texto).strip():
        return None
    t = unidecode(str(texto)).lower()
    t = re.sub(r'[^a-z\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t if t else None

def nombre_canonico(texto):
    norm = normalizar_nombre(texto)
    if norm is None:
        return None
    return ' '.join(sorted(norm.split()))

def normalizar_estado(estado):
    if pd.isna(estado) or not str(estado).strip():
        return None
    e = unidecode(str(estado)).upper().strip()
    return MAPA_ESTADOS.get(e, e)

def parsear_fecha(valor):
    if pd.isna(valor) or str(valor).strip().lower() == 'indeterminada':
        return pd.NaT
    for fmt in ('%d/%m/%Y', '%Y-%m-%dT%H:%M:%S'):
        try:
            return pd.to_datetime(str(valor), format=fmt)
        except ValueError:
            pass
    return pd.to_datetime(valor, errors='coerce', dayfirst=True)

def extraer_xml_lista(texto):
    if pd.isna(texto) or not str(texto).strip():
        return None
    elementos = re.findall(r'<element>([^<]+)</element>', str(texto))
    if not elementos:
        return None
    return ' | '.join(e.strip() for e in elementos)

print('Funciones de normalización definidas')

Funciones de normalización definidas


In [7]:
# 2.2 Aplicar limpieza al INE
ine = df_ine_raw.copy()
ine.columns = ine.columns.str.strip()

ine['nombre']             = ine['Nombre'].apply(normalizar_nombre)
ine['entidad']            = ine['Entidad Federativa'].apply(normalizar_estado)
ine['fecha_resolucion']   = ine['Fecha De La Resolución'].apply(parsear_fecha)
ine['permanencia_fecha']  = ine['Permanencia'].apply(parsear_fecha)
ine['tipo_violencia']     = ine['Tipo de violencia'].apply(extraer_xml_lista)
ine['medidas_reparacion'] = ine['Medidas de reparacion'].apply(extraer_xml_lista)

print(f'INE limpio: {ine.shape}')
ine[['nombre', 'entidad', 'fecha_resolucion',
     'permanencia_fecha', 'medidas_reparacion', 'tipo_violencia']].tail(5)

INE limpio: (530, 31)


,nombre,entidad,fecha_resolucion,permanencia_fecha,medidas_reparacion,tipo_violencia
525,yolanda adelaida santos montano,OAXACA,2021-06-11,2027-09-29,Medidas de no repetición | Disculpa pública | Medida de rehabilitación | Medidas de satisfacción...,No lo precisa
526,yolanda adelaida santos montano,OAXACA,2021-06-11,2027-09-29,Medidas de no repetición | Disculpa pública | Medidas de rehabilitación | Medidas de satisfacción,No lo precisa
527,yolanda adelaida santos montano,OAXACA,2022-05-04,2028-05-05,No impone,No lo precisa
528,yoshio cesar ramirez castillo,OAXACA,2024-09-20,2029-06-04,Disculpa pública | Medidas de no repetición | Medidas de no rehabilitación,Verbal | Psicológica | Simbólica
529,yuridia pineda ordaz,OAXACA,2023-02-08,2029-06-01,Disculpa pública | Medidas de no repetición | Medidas de rehabilitación | Medidas de satisfacción,Psicológica | Simbólica


In [8]:
# 2.3 Filtrar registros 2020-2022
ine = ine[ine['fecha_resolucion'].dt.year.between(2020, 2022)].reset_index(drop=True)
print(f'INE (2020-2022): {ine.shape}')

INE (2020-2022): (162, 31)


### 2.2 BANAVIM

In [9]:
# 2.4 Limpieza BANAVIM — hoja AGRESORES (2020-2022)
FLAGS_ARMAS  = ['Chacos','Macanas','OtraArmaBlanca','ObjetoPunzoCortante',
                'Machete','Proyectil','ArmaFuegoCorta','ArmaFuegoLarga','OtraFuegoLarga']
FLAGS_DROGAS = ['Droga_Alcohol','Droga_DrogaPorIndicación Médica','Droga_Drogas Ilegales']

def limpiar_bv_agresores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['estado']          = df['Estado donde reside'].apply(normalizar_estado)
    df['municipio']       = df['Municipio donde reside'].astype(str).str.strip().replace({'nan': None})
    df['sexo']            = df['Sexo'].astype(str).str.strip().str.title().replace({'Nan': None, 'nan': None})
    df['edad']            = pd.to_numeric(df['Edad del Agresor'], errors='coerce')
    df['escolaridad']     = df['Escolaridad'].astype(str).str.strip().replace({'nan': None})
    df['estado_civil']    = df['Estado Civil'].astype(str).str.strip().replace({'nan': None})
    df['vinculo_victima'] = df['Relación o vículo con la víctima'].astype(str).str.strip().replace({'nan': None})
    df['fecha_registro']  = pd.to_datetime(df['Fecha de registro'], errors='coerce')
    for col in FLAGS_DROGAS + FLAGS_ARMAS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

bv_2020 = limpiar_bv_agresores(bv_2020)
bv_2021 = limpiar_bv_agresores(bv_2021)
bv_2022 = limpiar_bv_agresores(bv_2022)
bv = pd.concat([bv_2020, bv_2021, bv_2022], ignore_index=True)

print(f'BANAVIM 2020 limpio: {bv_2020.shape}')
print(f'BANAVIM 2021 limpio: {bv_2021.shape}')
print(f'BANAVIM 2022 limpio: {bv_2022.shape}')
print(f'BANAVIM total (bv) : {bv.shape}')
print()
campos_clave = ['estado','municipio','sexo','edad','escolaridad','estado_civil','vinculo_victima']
print('Completitud de campos clave:')
print((1 - bv[campos_clave].isnull().mean()).mul(100).round(1).to_string())

BANAVIM 2020 limpio: (265760, 36)
BANAVIM 2021 limpio: (285984, 36)
BANAVIM 2022 limpio: (254348, 36)
BANAVIM total (bv) : (806092, 36)

Completitud de campos clave:
0
estado             100.0
municipio          100.0
sexo                82.4
edad                82.1
escolaridad         83.2
estado_civil        49.9
vinculo_victima     83.8


In [10]:
# 2.5 Auditoría de codificación rota en BANAVIM
PATRON_MOJIBAKE = r'[ÃÂ�]'
cols_texto_bv = bv.select_dtypes(include='object').columns.tolist()

reporte_mojibake = []
for col in cols_texto_bv:
    serie = bv[col].astype(str)
    mask  = serie.str.contains(PATRON_MOJIBAKE, regex=True, na=False)
    reporte_mojibake.append({
        'columna': col,
        'filas_afectadas': int(mask.sum()),
        'pct_afectado': round(mask.mean() * 100, 4),
        'valores_unicos_afectados': bv.loc[mask, col].nunique(dropna=True)
    })

reporte_mojibake = (
    pd.DataFrame(reporte_mojibake)
    .sort_values('filas_afectadas', ascending=False)
    .reset_index(drop=True)
)
display(reporte_mojibake)

,columna,filas_afectadas,pct_afectado,valores_unicos_afectados
0,Relación o vículo con la víctima,305694,37.9230,6
1,vinculo_victima,305694,37.9230,6
2,municipio,296001,36.7205,646
3,Municipio donde reside,296001,36.7205,646
4,Estado donde reside,213569,26.4944,7
5,Estado Civil,61713,7.6558,1
6,estado_civil,61713,7.6558,1
7,escolaridad,10943,1.3575,3
8,Escolaridad,10943,1.3575,3
9,Cual Droga,1634,0.2027,1


In [11]:
# 2.6 Ejemplos de valores con codificación rota
ejemplos_mojibake = []
for col in cols_texto_bv:
    mask = bv[col].astype(str).str.contains(PATRON_MOJIBAKE, regex=True, na=False)
    if mask.any():
        vals = bv.loc[mask, col].dropna().astype(str).drop_duplicates().head(5)
        for v in vals:
            ejemplos_mojibake.append({'columna': col, 'valor_afectado': v})
display(pd.DataFrame(ejemplos_mojibake))

,columna,valor_afectado
0,Escolaridad,Carrera tÃ©cnica comercial
1,Escolaridad,MaestrÃ­a
2,Escolaridad,Estudios que no requieren validÃ©z oficial
3,Estado Civil,UniÃ³n libre
4,Estado donde reside,San Luis PotosÃ­
5,Estado donde reside,Ciudad de MÃ©xico
6,Estado donde reside,Estado de MÃ©xico
7,Estado donde reside,Nuevo LeÃ³n
8,Estado donde reside,QuerÃ©taro
9,Municipio donde reside,CosÃ­o


In [12]:
from ftfy import fix_text

# 2.7 Función para reparar codificación (mojibake → texto correcto)
def reparar_codificacion(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == '':
        return np.nan
    return fix_text(s)

In [13]:
# 2.8 Crear copia de BANAVIM con texto reparado
bv_fix = bv.copy()
for col in cols_texto_bv:
    bv_fix[col] = bv_fix[col].apply(reparar_codificacion)

bv_fix.to_pickle(DATA_DIR / 'banavim_fix_2020_2022.pkl')

print(f'bv original : {bv.shape}')
print(f'bv_fix      : {bv_fix.shape}')

bv original : (806092, 36)
bv_fix      : (806092, 36)


In [14]:
# 2.9 Verificación antes/después de reparación
comparacion_codificacion = []
for col in cols_texto_bv:
    antes   = bv[col].astype(str)
    despues = bv_fix[col].astype(str)
    mask_antes   = antes.str.contains(PATRON_MOJIBAKE, regex=True, na=False)
    mask_despues = despues.str.contains(PATRON_MOJIBAKE, regex=True, na=False)
    comparacion_codificacion.append({
        'columna': col,
        'filas_mojibake_antes':  int(mask_antes.sum()),
        'filas_mojibake_despues': int(mask_despues.sum()),
        'valores_modificados': int((antes != despues).sum()),
        'unicos_antes':  bv[col].nunique(dropna=True),
        'unicos_despues': bv_fix[col].nunique(dropna=True)
    })

display(
    pd.DataFrame(comparacion_codificacion)
    .sort_values('filas_mojibake_antes', ascending=False)
    .reset_index(drop=True)
)

,columna,filas_mojibake_antes,filas_mojibake_despues,valores_modificados,unicos_antes,unicos_despues
0,Relación o vículo con la víctima,305694,0,310966,23,23
1,vinculo_victima,305694,0,436621,23,23
2,municipio,296001,0,296023,1945,1945
3,Municipio donde reside,296001,0,296001,1945,1945
4,Estado donde reside,213569,0,213569,33,33
5,Estado Civil,61713,0,61713,10,10
6,estado_civil,61713,0,465786,10,10
7,escolaridad,10943,0,146315,13,13
8,Escolaridad,10943,0,10943,13,13
9,Cual Droga,1634,0,1634,3,3


In [15]:
# 2.10 Ejemplos antes/después de reparación
ejemplos_fix = []
for col in cols_texto_bv:
    tmp = pd.DataFrame({'columna': col, 'antes': bv[col], 'despues': bv_fix[col]})
    tmp = tmp[tmp['antes'].astype(str) != tmp['despues'].astype(str)].drop_duplicates()
    ejemplos_fix.append(tmp.head(5))
display(pd.concat(ejemplos_fix, ignore_index=True))

,columna,antes,despues
0,Escolaridad,Carrera tÃ©cnica comercial,Carrera técnica comercial
1,Escolaridad,MaestrÃ­a,Maestría
2,Escolaridad,Estudios que no requieren validÃ©z oficial,Estudios que no requieren validéz oficial
3,Estado Civil,UniÃ³n libre,Unión libre
4,Estado donde reside,San Luis PotosÃ­,San Luis Potosí
5,Estado donde reside,Ciudad de MÃ©xico,Ciudad de México
6,Estado donde reside,Estado de MÃ©xico,Estado de México
7,Estado donde reside,Nuevo LeÃ³n,Nuevo León
8,Estado donde reside,QuerÃ©taro,Querétaro
9,Municipio donde reside,CosÃ­o,Cosío


In [16]:
# 2.11 Guardar BANAVIM con codificación reparada
bv_fix.to_csv(
    DATA_DIR / 'banavim_fix_2020_2022.csv',
    index=False, encoding='utf-8-sig'
)
print('Guardado: banavim_fix_2020_2022.csv')

Guardado: banavim_fix_2020_2022.csv


## 3. Fusión de Datos

### 3.1 Consolidación del Golden Record INE

In [17]:
# Recomputar completitud INE (necesaria para descartar columnas con <20% completitud)
completitud_ine = (1 - df_ine_raw.isnull().mean()) * 100


In [18]:
# Eliminar columnas con menos del 20% de completitud
ine = ine.drop(columns=completitud_ine[completitud_ine < 20].index)
print(f'INE ajustado (columnas >=20% completitud): {ine.shape}')

# Guardar base de incidencias
ine.to_csv(
    DATA_DIR / 'ine_incidencias_2020_2022.csv',
    index=False, encoding='utf-8-sig'
)
print('Guardado: ine_incidencias_2020_2022.csv')

INE ajustado (columnas >=20% completitud): (162, 25)
Guardado: ine_incidencias_2020_2022.csv


In [19]:
# 3.1 Campos invariantes vs. campos por incidencia
CAMPOS_INVARIANTES = [
    'Sexo', 'Ámbito Territorial', 'Entidad Federativa', 'Municipio',
    'Relación Con La Víctima', 'entidad', 'nombre',
]
CAMPOS_POR_INCIDENCIA = [
    'Número De Expediente', 'Incidencia', 'Órgano Resolutor', 'Conducta',
    'Sanción', 'Permanencia', 'Reincidencia De La Conducta', 'Resolución Penal',
    'Analizó Modo Honesto De Vivir', 'Cumple Modo Honesto De Vivir',
    'Documento Enlace', 'Enlace Utce Temporalidad', 'Modalidad de violencia',
    'fecha_resolucion', 'permanencia_fecha', 'tipo_violencia', 'medidas_reparacion',
]
print(f'Campos invariantes    : {len(CAMPOS_INVARIANTES)}')
print(f'Campos por incidencia : {len(CAMPOS_POR_INCIDENCIA)}')

Campos invariantes    : 7
Campos por incidencia : 17


In [20]:
# 3.2 Función de consolidación (golden record por persona)
def consolidar_persona(df_grupo: pd.DataFrame) -> dict:
    """
    Recibe todas las filas de un mismo Nombre y devuelve un dict con:
    - Campos invariantes : moda del valor a lo largo de las incidencias.
    - Campos por incidencia : lista ordenada cronológicamente.
    - Campos derivados : n_incidencias, es_reincidente.
    """
    df_grupo = df_grupo.sort_values('fecha_resolucion', na_position='last')
    gr = {}
    gr['Nombre'] = max(df_grupo['Nombre'].dropna().tolist(),
                       key=lambda x: len(str(x)), default=None)
    for col in CAMPOS_INVARIANTES:
        if col not in df_grupo.columns:
            gr[col] = None; continue
        vals = df_grupo[col].dropna()
        gr[col] = max(vals.mode().tolist(), key=lambda x: len(str(x))) if len(vals) > 0 else None
    for col in CAMPOS_POR_INCIDENCIA:
        if col not in df_grupo.columns:
            gr[col] = []; continue
        gr[col] = [None if pd.isna(v) else v for v in df_grupo[col].tolist()]
    gr['n_incidencias']  = len(df_grupo)
    gr['es_reincidente'] = len(df_grupo) > 1
    return gr

print('Función consolidar_persona definida')

Función consolidar_persona definida


In [21]:
# 3.3 Aplicar consolidación por Nombre
ine_con_nombre = ine[ine['Nombre'].notna()].copy()
ine_sin_nombre = ine[ine['Nombre'].isna()].copy()

registros_golden = []
for nombre, df_grupo in ine_con_nombre.groupby('Nombre', sort=False):
    registros_golden.append(consolidar_persona(df_grupo))

df_golden = pd.DataFrame(registros_golden)

print(f'Registros INE originales con nombre   : {len(ine_con_nombre):,}')
print(f'Individuos únicos tras consolidación  : {len(df_golden):,}')
print(f'Registros consolidados (reincidencias): {len(ine_con_nombre) - len(df_golden):,}')
print(f'Individuos reincidentes               : {df_golden["es_reincidente"].sum():,}')

Registros INE originales con nombre   : 162
Individuos únicos tras consolidación  : 140
Registros consolidados (reincidencias): 22
Individuos reincidentes               : 12


In [22]:
# 3.4 Vista previa de individuos reincidentes
display(
    df_golden[df_golden['es_reincidente']]
    [['Nombre', 'entidad', 'n_incidencias', 'Conducta', 'Sanción', 'fecha_resolucion']]
    .head(6)
)

,Nombre,entidad,n_incidencias,Conducta,Sanción,fecha_resolucion
25,CARLOS MARIO CORNELIO CORNELIO,TABASCO,2,[Obstaculizar la entrega de licencia para la separación del cargo de delegada municipal para par...,"[Multa económica, Multa económica]","[2022-02-28 00:00:00, 2022-03-31 00:00:00]"
28,DARWIN FÉLIX LÓPEZ,TABASCO,2,[Obstaculizar la entrega de licencia para separarse del cargo de delegada municipal para partici...,"[Multa económica, Multa económica]","[2022-02-28 00:00:00, 2022-03-31 00:00:00]"
36,ELÍ MARTÍNEZ LÓPEZ,OAXACA,2,[Conductas realizadas de manera sistemática dirigidas a sobajar a la actora de la oportunidad de...,"[Ninguna, Ninguna]","[2021-09-10 00:00:00, 2022-12-16 00:00:00]"
41,ERNESTO RUIZ FLANDES,VERACRUZ,10,[Conducta reiterada de convocar a la víctima a sesiones de cabildo sin la documentación completa...,"[Ninguna, Ninguna, Ninguna, Ninguna, Ninguna, Ninguna, Ninguna, Ninguna, Ninguna, Ninguna]","[2020-11-09 00:00:00, 2020-11-26 00:00:00, 2020-12-30 00:00:00, 2021-01-14 00:00:00, 2021-02-09 ..."
54,GILDARDO ZENTENO MORENO,CHIAPAS,2,[Reiteración y sistematización de la obstaculización e invisibilización del ejercicio del cargo ...,"[Ninguna, Ninguna]","[2021-05-01 00:00:00, 2021-11-10 00:00:00]"
57,HERMAS CORTÉS GARCÍA,VERACRUZ,2,"[Omisión de contestar diversos oficios, así como convocar a sesiones sin tomar en cuenta el esta...","[Amonestación pública, Ninguna]","[2021-03-03 00:00:00, 2021-05-12 00:00:00]"


In [23]:
# 3.5 Ejemplo detallado: persona con más incidencias
max_idx = df_golden['n_incidencias'].idxmax()
ej = df_golden.loc[max_idx]
print(f"Nombre      : {ej['Nombre']}")
print(f"Entidad     : {ej['entidad']}")
print(f"Incidencias : {ej['n_incidencias']}")
print()
for i, (cond, sanc, fecha) in enumerate(
    zip(ej['Conducta'], ej['Sanción'], ej['fecha_resolucion']), 1
):
    print(f'  [{i}] Fecha  : {fecha}')
    print(f'       Sanción : {sanc}')
    print(f'       Conducta: {str(cond)[:120]}')
    print()

Nombre      : ERNESTO RUIZ FLANDES
Entidad     : VERACRUZ
Incidencias : 10

  [1] Fecha  : 2020-11-09 00:00:00
       Sanción : Ninguna
       Conducta: Conducta reiterada de convocar a la víctima a sesiones de cabildo sin la documentación completa

  [2] Fecha  : 2020-11-26 00:00:00
       Sanción : Ninguna
       Conducta: Conducta reiterada de convocar a la víctima a sesiones de cabildo sin la documentación completa

  [3] Fecha  : 2020-12-30 00:00:00
       Sanción : Ninguna
       Conducta: Vulneración al derecho del ejercicio del cargo, al iniciar una sesión de Cabildo 15 minutos antes de la hora establecida

  [4] Fecha  : 2021-01-14 00:00:00
       Sanción : Ninguna
       Conducta: 1.-No se citó a la actora a las sesiones de cabildo con la documentación completa; 2.- Omisión de contestar peticiones d

  [5] Fecha  : 2021-02-09 00:00:00
       Sanción : Ninguna
       Conducta: Convocar a la víctima a sesiones sin la documentación correspondiente

  [6] Fecha  : 2021-04-13 00:0

In [24]:
# Exportar df_golden para fusion.ipynb
with open(DATA_DIR / 'df_golden.pkl', 'wb') as f:
    pickle.dump(df_golden, f)
print('Guardado: df_golden.pkl')


Guardado: df_golden.pkl
